In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from xgboost import XGBClassifier

## Layer 1 — Base Model (600k lexical features)

In [ ]:
df_full = pd.read_csv('..\data\Full_lexical_features.csv')
print('Shape:', df_full.shape)
print('Class distribution:')
print(df_full['type'].value_counts())

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Hanna Hahn\AppData\Local\Temp\ipykernel_10256\785583806.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_full = pd.read_csv('..\data\Full_lexical_features.csv')


Shape: (651191, 21)
Class distribution:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


C:\Users\Hanna Hahn\AppData\Local\Temp\ipykernel_10256\785583806.py:1: DtypeWarning: Columns (0: domain) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv('..\data\Full_lexical_features.csv')


In [ ]:
X_full = df_full.drop(columns=['url', 'domain', 'path', 'type'])
y_full = df_full['type']

le = LabelEncoder()
y_full_encoded = le.fit_transform(y_full)
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full_encoded, test_size=0.2, random_state=42, stratify=y_full_encoded
)

Label mapping: {'benign': np.int64(0), 'defacement': np.int64(1), 'malware': np.int64(2), 'phishing': np.int64(3)}


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest (600k) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Macro F1: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

=== Random Forest (600k) ===
Accuracy: 0.9414
Macro F1: 0.9287
              precision    recall  f1-score   support

      benign       0.98      0.95      0.96     85621
  defacement       0.97      0.99      0.98     19292
     malware       0.98      0.94      0.96      6504
    phishing       0.77      0.87      0.82     18822

    accuracy                           0.94    130239
   macro avg       0.92      0.94      0.93    130239
weighted avg       0.95      0.94      0.94    130239



In [ ]:
# XGBoost
xgb = XGBClassifier(
    objective='multi:softmax', num_class=len(le.classes_),
    n_estimators=400, learning_rate=0.1, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('=== XGBoost (600k) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'Macro F1: {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

=== XGBoost (600k) ===
Accuracy: 0.9453
Macro F1: 0.9251
              precision    recall  f1-score   support

      benign       0.96      0.97      0.97     85621
  defacement       0.96      0.99      0.97     19292
     malware       0.98      0.92      0.95      6504
    phishing       0.85      0.78      0.81     18822

    accuracy                           0.95    130239
   macro avg       0.94      0.92      0.93    130239
weighted avg       0.94      0.95      0.94    130239



In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_full.columns,
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)

print('Top 15 features:')
print(feature_importance.head(15))

Top 15 features:
                feature  importance
1         domain_length    0.144686
16       domain_entropy    0.139026
6           num_slashes    0.117603
7            num_digits    0.087327
2           path_length    0.079745
15       num_subdomains    0.073208
8           num_letters    0.072159
9     num_special_chars    0.071526
0            url_length    0.059461
3              num_dots    0.047630
4           num_hyphens    0.033598
10               has_ip    0.028076
5       num_underscores    0.023878
14  has_suspicious_word    0.020624
11        has_at_symbol    0.001009


## Layer 2 — Enriched Model (2k with passive OSINT)

In [ ]:
df_passive = pd.read_csv('..\data\Passive_features.csv')
print('Shape:', df_passive.shape)
print('Class distribution:')
print(df_passive['type'].value_counts())

Shape: (2000, 28)
Class distribution:
type
benign        1445
defacement     360
phishing       137
malware         58
Name: count, dtype: int64


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Hanna Hahn\AppData\Local\Temp\ipykernel_10256\1826556858.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_passive = pd.read_csv('..\data\Passive_features.csv')


In [ ]:
X_passive = df_passive.drop(columns=['url', 'domain', 'path', 'type', 'ip', 'asn_org', 'country', 'region'])
y_passive = df_passive['type']

le2 = LabelEncoder()
y_passive_encoded = le2.fit_transform(y_passive)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_passive, y_passive_encoded, test_size=0.2, random_state=42, stratify=y_passive_encoded
)

print('New passive features added:', ['domain_age_days', 'is_dga_like', 'suspicious_domain'])
print('Total features:', X_passive.shape[1])

New passive features added: ['domain_age_days', 'is_dga_like', 'suspicious_domain']
Total features: 20


In [ ]:
# XGBoost on enriched data
xgb2 = XGBClassifier(
    objective='multi:softmax', num_class=len(le2.classes_),
    n_estimators=400, learning_rate=0.1, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
xgb2.fit(X_train_p, y_train_p)
y_pred_xgb2 = xgb2.predict(X_test_p)

print('=== XGBoost (2k + Passive OSINT) ===')
print(f'Accuracy: {accuracy_score(y_test_p, y_pred_xgb2):.4f}')
print(f'Macro F1: {f1_score(y_test_p, y_pred_xgb2, average="macro"):.4f}')
print(classification_report(y_test_p, y_pred_xgb2, target_names=le2.classes_))

=== XGBoost (2k + Passive OSINT) ===
Accuracy: 0.9650
Macro F1: 0.9140
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.88      0.94      0.91        72
     malware       1.00      0.75      0.86        12
    phishing       1.00      0.81      0.90        27

    accuracy                           0.96       400
   macro avg       0.97      0.88      0.91       400
weighted avg       0.97      0.96      0.96       400



In [ ]:
print('=== MODEL COMPARISON ===')
print(f'Base RandomForest(600k) Macro F1: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'Base XGBoosting(600k) Macro F1: {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'Enriched XGBoosting (2k) Macro F1: {f1_score(y_test_p, y_pred_xgb2, average="macro"):.4f}')

=== MODEL COMPARISON ===
Base RandomForest(600k) Macro F1: 0.9287
Base XGBoosting(600k) Macro F1: 0.9251
Enriched XGBoosting (2k) Macro F1: 0.9140


### Saving models

In [ ]:
import joblib
from pathlib import Path

Path('..\Model').mkdir(exist_ok=True)

joblib.dump(rf, '..\Model\RandomForest_model.pkl')
joblib.dump(xgb, '..\Model\xgb_model.pkl')
joblib.dump(le, '..\Model\label_encoder.pkl') 

print('Models saved!')